# Train the Task 1 BERT classifier on a Colab GPU

Use this when your own machine cannot fine-tune BERT (for example an 8 GB Mac).
It trains exactly what notebook 06 trains (`src/classify/bert.py`, frozen split, seed 42) and gives you a zip of the model.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Training takes about 10-15 minutes. At the end the browser downloads `bert_intent.zip` (about 400 MB).
3. On your machine, unzip it into `outputs/bert_intent/` and run notebook 06: it detects the model and skips training.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
BRANCH = "dev/zfang75"   # branch that holds the frozen split
!rm -rf /content/banking-support-agent
!git clone -q -b {BRANCH} https://github.com/sh1or1tama1/banking-support-agent.git /content/banking-support-agent
%cd /content/banking-support-agent
!pip -q install "accelerate>=1.1" "datasets>=2.18" "pyarrow>=15"
!git log --oneline -1

In [ ]:
import sys, time
sys.path.insert(0, "/content/banking-support-agent")
import pandas as pd, transformers
from src import config as C
from src.utils import set_seed
from src.classify.bert import train_bert
set_seed()
df = pd.read_parquet(C.SPLITS_FILE)
train, val = df[df["split"] == "train"], df[df["split"] == "val"]
print(len(train), "train |", len(val), "val | transformers", transformers.__version__)
t0 = time.time()
model, tok, labels, history = train_bert(train, val)
print(f"training took {(time.time() - t0) / 60:.1f} min")
history.to_csv(C.OUTPUTS / "bert_intent" / "training_history.csv", index=False)
history[["epoch", "eval_accuracy", "eval_macro_f1", "eval_loss"]]

In [ ]:
# Zip only the final model files (not the checkpoint-* folders with optimizer state) and download
import os, zipfile
from google.colab import files
src = C.OUTPUTS / "bert_intent"
keep = [f for f in sorted(os.listdir(src)) if (src / f).is_file()]
print(keep)
with zipfile.ZipFile("/content/bert_intent.zip", "w", zipfile.ZIP_STORED) as z:
    for f in keep:
        z.write(src / f, arcname=f)
print(f"{os.path.getsize('/content/bert_intent.zip') / 1e6:.0f} MB")
files.download("/content/bert_intent.zip")